In [1]:
# 1. CÀI ĐẶT THƯ VIỆN CẦN THIẾT
!pip install roboflow ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 102.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.6 MB/s eta 0:00:00


In [2]:
import torch

print("===== GPU INFORMATION =====")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("CUDA version:", torch.version.cuda)
    print("VRAM total: {:.2f} GB".format(
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    ))
else:
    print("GPU: Not available")

===== GPU INFORMATION =====
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU count: 2
CUDA version: 12.8
VRAM total: 14.56 GB


In [3]:
# 2. TẢI DATASET VÀ IN THÔNG TIN CHI TIẾT
import os
import yaml
from roboflow import Roboflow

# 1. Tải Dataset từ Roboflow
rf = Roboflow(api_key="7cluP5yURKpTZxVV8Wh0") 
project = rf.workspace("material-identification").project("garbage-classification-3")
version = project.version(2)
dataset = version.download("yolov8")

# 2. Đọc nội dung file data.yaml
yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path, 'r') as file:
    data_yaml = yaml.safe_load(file)

# 3. Đếm số lượng ảnh trong các tập train / val / test
def count_images(folder_path):
    if os.path.exists(folder_path):
        return len([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    return 0

train_count = count_images(os.path.join(dataset.location, "train/images"))
val_count   = count_images(os.path.join(dataset.location, "valid/images"))
test_count  = count_images(os.path.join(dataset.location, "test/images"))
total_images = train_count + val_count + test_count

# 4. In thông tin chi tiết ra màn hình
print("\n" + "="*50)
print("THÔNG TIN TẬP DỮ LIỆU (DATASET METADATA)")
print("="*50)
print(f"Vị trí lưu dataset: {dataset.location}")
print(f"Số lượng class (nc): {data_yaml.get('nc', len(data_yaml.get('names', [])))}")
print(f"Danh sách các class: {data_yaml.get('names')}")
print("-"*50)
print(f"Tập Train : {train_count:>5} ảnh ({train_count/total_images*100:.1f}%)")
print(f"Tập Valid : {val_count:>5} ảnh ({val_count/total_images*100:.1f}%)")
print(f"Tập Test  : {test_count:>5} ảnh ({test_count/total_images*100:.1f}%)")
print(f"TỔNG CỘNG : {total_images:>5} ảnh")
print("="*50)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to GARBAGE-CLASSIFICATION-3-2 in yolov8:: 100%|██████████| 20940/20940 [00:01<00:00, 11322.28it/s]


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

THÔNG TIN TẬP DỮ LIỆU (DATASET METADATA)
Vị trí lưu dataset: /kaggle/working/GARBAGE-CLASSIFICATION-3-2
Số lượng class (nc): 6
Danh sách các class: ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
--------------------------------------------------
Tập Train :  7324 ảnh (70.0%)
Tập Valid :  2098 ảnh (20.0%)
Tập Test  :  1042 ảnh (10.0%)
TỔNG CỘNG : 10464 ảnh


In [4]:
# CELL 3: FINETUNE RT-DETR-X
from ultralytics import RTDETR

# Tải pre-trained RT-DETR-X (Extra Large)
model = RTDETR("rtdetr-x.pt")

# Tiến hành huấn luyện
train_results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=40,             # 50 epochs giúp Transformer hội tụ hoàn hảo
    patience=20,
    close_mosaic=5,
    imgsz=640,
    batch=8,               # Chuẩn cho GPU T4 (không bị lỗi CUDA Out of Memory)
    workers=2,
    optimizer='AdamW',     # Optimizer tối ưu nhất cho họ Transformer
    lr0=0.0001,            # Learning rate chuẩn cho finetune Transformer
    device=0,              # Sử dụng GPU Kaggle
    name="RT_DETR_X_Garbage",
    save=True,
    plots=True
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/GARBAGE-CLASSIFICATION-3-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=RT_DETR

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       1/40      9.27G     0.9847        1.1     0.6864         11        640: 100% ━━━━━━━━━━━━ 916/916 1.2it/s 12:421.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 1.9it/s 1:090.5sss
                   all       2098      18916      0.365      0.139     0.0146    0.00764

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/40      9.46G     0.5316      1.303     0.2447         80        640: 0% ──────────── 0/916  0.8s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       2/40       9.5G     0.6484      1.301     0.3748         22        640: 100% ━━━━━━━━━━━━ 916/916 1.2it/s 12:260.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 2.1it/s 1:040.5sss
                   all       2098      18916      0.221      0.176     0.0487     0.0244

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/40      9.54G     0.5105      1.224     0.3828         77        640: 0% ──────────── 0/916  0.9s

...

      38/40      9.48G      0.381     0.4489     0.2312          7        640: 100% ━━━━━━━━━━━━ 916/916 1.2it/s 12:230.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 2.0it/s 1:040.5sss
                   all       2098      18916      0.646      0.552      0.594      0.416

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      39/40      9.53G      1.329     0.3252     0.3493        128        640: 0% ──────────── 0/916  0.9s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      39/40      9.56G     0.3751     0.4364     0.2236         50        640: 100% ━━━━━━━━━━━━ 916/916 1.2it/s 12:230.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 2.0it/s 1:050.5sss
                   all       2098      18916      0.655      0.547      0.595      0.416

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      40/40      9.19G     0.5145     0.4521     0.1722         96        640: 0% ──────────── 0/916  0.8s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      40/40      9.25G     0.3776     0.4344       0.22         15        640: 100% ━━━━━━━━━━━━ 916/916 1.2it/s 12:230.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 2.0it/s 1:040.5sss
                   all       2098      18916      0.658      0.551      0.599       0.42

40 epochs completed in 9.025 hours.
Optimizer stripped from /kaggle/working/runs/detect/RT_DETR_X_Garbage/weights/last.pt, 135.5MB
Optimizer stripped from /kaggle/working/runs/detect/RT_DETR_X_Garbage/weights/best.pt, 135.5MB

Validating /kaggle/working/runs/detect/RT_DETR_X_Garbage/weights/best.pt...
Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-x summary: 387 layers, 65,479,766 parameters, 0 gradients, 224.7 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 2.2it/s 1:010.4sss
                   all       2098      18916     

In [9]:
# 5. ĐÓNG GÓI TOÀN BỘ KẾT QUẢ & TRỌNG SỐ RT-DETR-X 
import os
import shutil
import glob
from pathlib import Path

# 1. Định nghĩa thư mục kết quả của RT-DETR-X
RUN_DIR = Path("runs/detect/RT_DETR_X_Garbage")
BUNDLE_DIR = "/kaggle/working/RT_DETR_X_Bundle"

# Tạo mới thư mục gom gói (nếu đã có thì xóa làm mới)
if os.path.exists(BUNDLE_DIR):
    shutil.rmtree(BUNDLE_DIR)
os.makedirs(BUNDLE_DIR, exist_ok=True)

print("=" * 60)
print("ĐANG TIẾN HÀNH GOM CÁC THÀNH PHẦN QUAN TRỌNG...")
print("=" * 60)

# 2. Gom các file weights (best.pt và last.pt)
weights_bundle_dir = os.path.join(BUNDLE_DIR, "weights")
os.makedirs(weights_bundle_dir, exist_ok=True)

weights_src_dir = RUN_DIR / "weights"
for weight_name in ["best.pt", "last.pt"]:
    src = weights_src_dir / weight_name
    if src.exists():
        shutil.copy(src, weights_bundle_dir)
        size_mb = src.stat().st_size / (1024 * 1024)
        print(f"Đã gom: weights/{weight_name} ({size_mb:.1f} MB)")
    else:
        print(f"Không tìm thấy: {weight_name}")

# 3. Gom toàn bộ ảnh đồ thị, file CSV kết quả, file config YAML để làm Báo Cáo
plot_files = (
    glob.glob(f"{RUN_DIR}/*.png") + 
    glob.glob(f"{RUN_DIR}/*.jpg") + 
    glob.glob(f"{RUN_DIR}/*.csv") + 
    glob.glob(f"{RUN_DIR}/*.yaml")
)

for file_path in plot_files:
    shutil.copy(file_path, BUNDLE_DIR)

print(f"Đã gom: {len(plot_files)} file biểu đồ, cấu hình & bảng số liệu CSV")

# 4. Nén toàn bộ thư mục BUNDLE thành 1 file ZIP duy nhất
ZIP_FILE = "/kaggle/working/RT_DETR_X_Garbage_pack.zip"
!zip -r {ZIP_FILE} {BUNDLE_DIR} > /dev/null

zip_size_mb = os.path.getsize(ZIP_FILE) / (1024 * 1024)

print("\n" + "=" * 60)
print(f"ĐÃ ĐÓNG GÓI XONG FILE: RT_DETR_X_Garbage_pack.zip ({zip_size_mb:.1f} MB)")
print("Vị trí file: /kaggle/working/RT_DETR_X_Garbage_pack.zip")
print("=" * 60)

ĐANG TIẾN HÀNH GOM CÁC THÀNH PHẦN QUAN TRỌNG...
Đã gom: weights/best.pt (129.2 MB)
Đã gom: weights/last.pt (129.2 MB)
Đã gom: 22 file biểu đồ, cấu hình & bảng số liệu CSV

ĐÃ ĐÓNG GÓI XONG FILE: RT_DETR_X_Garbage_pack.zip (246.1 MB)
Vị trí file: /kaggle/working/RT_DETR_X_Garbage_pack.zip


In [16]:
from ultralytics import RTDETR

model = RTDETR("runs/detect/RT_DETR_X_Garbage/weights/best.pt")

metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
    conf=0.25,
    iou=0.5,
    plots=True
)

# In các chỉ số chính
print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP@0.5   :", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

# F1 có thể tự tính
precision = metrics.box.mp
recall = metrics.box.mr
f1 = 2 * precision * recall / (precision + recall + 1e-16)
print("F1-score  :", f1)

# AP từng class
print("\nAP từng class:")
for i, name in enumerate(model.names.values()):
    print(f"{name:15s}: {metrics.box.maps[i]:.4f}")

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-x summary: 387 layers, 65,479,766 parameters, 0 gradients, 224.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 908.0±588.8 MB/s, size: 25.7 KB)
val: Scanning /kaggle/working/GARBAGE-CLASSIFICATION-3-2/test/labels.cache... 1042 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1042/1042 208.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 66/66 1.1s/it 1:101.0sss
                   all       1042       3563      0.582      0.527      0.497      0.388
         BIODEGRADABLE          8         49      0.211      0.265      0.138     0.0684
             CARDBOARD         11         20      0.244       0.45      0.284      0.264
                 METAL        172        533      0.865      0.702      0.706      0.596
                 PAPER        515       1376      0.786      0.575      0.651      0.518
              

In [18]:
import numpy as np
from pathlib import Path

# ===== CẤU HÌNH =====
DATA_ROOT = Path(dataset.location)
PRED_DIR  = Path("runs/detect/RT_DETR_X_test_pred/labels")   # thư mục prediction
GT_DIR    = DATA_ROOT / "test" / "labels"

IOU_THR   = 0.50
CONF_THR  = 0.25

names = ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']

def read_yolo_txt(path, pred=False):
    cols = 6 if pred else 5
    if not path.exists():
        return np.zeros((0, cols))
    rows = []
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if parts:
            rows.append([float(x) for x in parts[:cols]])
    if not rows:
        return np.zeros((0, cols))
    arr = np.array(rows, dtype=float)
    if pred and arr.shape[1] == 5:
        arr = np.c_[arr, np.ones(len(arr))]
    return arr

def xywh_to_xyxy(x):
    y = x.copy()
    y[:, 0] = x[:, 0] - x[:, 2] / 2
    y[:, 1] = x[:, 1] - x[:, 3] / 2
    y[:, 2] = x[:, 0] + x[:, 2] / 2
    y[:, 3] = x[:, 1] + x[:, 3] / 2
    return np.clip(y, 0, 1)

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

# ===== TÍNH MEAN IoU =====
matched_ious = []

image_stems = [p.stem for p in (DATA_ROOT / "test" / "images").glob("*") 
               if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

for stem in image_stems:
    gt = read_yolo_txt(GT_DIR / f"{stem}.txt", pred=False)
    pred = read_yolo_txt(PRED_DIR / f"{stem}.txt", pred=True)

    if len(gt) == 0 or len(pred) == 0:
        continue

    # Lọc prediction theo confidence
    pred = pred[pred[:, 5] >= CONF_THR]
    if len(pred) == 0:
        continue

    gt_boxes = xywh_to_xyxy(gt[:, 1:5])
    pred_boxes = xywh_to_xyxy(pred[:, 1:5])
    pred_cls = pred[:, 0].astype(int)
    gt_cls = gt[:, 0].astype(int)

    # Chỉ match cùng class
    for c in np.unique(gt_cls):
        gt_c = gt_boxes[gt_cls == c]
        pred_c = pred_boxes[pred_cls == c]

        if len(gt_c) == 0 or len(pred_c) == 0:
            continue

        ious = box_iou(pred_c, gt_c)
        used = np.zeros(len(gt_c), dtype=bool)

        # Greedy matching theo IoU cao nhất
        for i in range(len(pred_c)):
            if ious[i].max() < IOU_THR:
                continue
            j = ious[i].argmax()
            if not used[j]:
                matched_ious.append(ious[i, j])
                used[j] = True

mean_iou = np.mean(matched_ious) if matched_ious else 0.0
print(f"Mean matched IoU (conf≥{CONF_THR}, IoU≥{IOU_THR}): {mean_iou:.4f}")
print(f"Số lượng box được match: {len(matched_ious)}")

Mean matched IoU (conf≥0.25, IoU≥0.5): 0.8482
Số lượng box được match: 2744
